# Chapter 10: Walking in Circles

## Holonomy, parallel transport, and curvature as a bivector

Imagine you are standing at the North Pole, holding an arrow pointing south toward London. You walk south to London, keeping the arrow pointing the same direction relative to your path (this is **parallel transport**). Then you walk along the equator to, say, Quito, Ecuador — still keeping the arrow parallel to your path. Finally, you walk north back to the North Pole.

When you arrive, the arrow is **no longer pointing toward London**. It has been rotated, even though you never deliberately turned it. The rotation of the arrow after a closed loop is called **holonomy**, and it measures the **curvature** of the surface you walked on.

The same idea applies to the transformer's representation space. We have a 2D grid:
- **Layer axis** (vertical): $l = 0, 1, \ldots, L-1$
- **Token axis** (horizontal): $t = 0, 1, \ldots, T-1$

At each grid point $(l, t)$, the model produces a hidden-state vector $h^{(l)}_t$. Moving along the grid — from one layer to the next, or from one token position to the next — produces a rotation (a rotor). If we compose rotors around a closed rectangular loop (a **plaquette**), we get the **holonomy rotor** $R_{\text{loop}}$.

The key insight from geometric algebra:

- **Scalar curvature**: $\|R_{\text{loop}} - I\|_F$ tells you *how much* curvature there is.
- **Curvature bivector**: $\log(R_{\text{loop}})$ tells you *in which plane* the curvature lives.

The bivector of the holonomy points in the direction of curvature — information that is completely lost when you only report a scalar norm.

**What you will learn:**
- How parallel transport works on the layer-time grid
- How to compute holonomy around a plaquette
- How to read the curvature bivector's principal plane
- How curvature concentrates in specific layers and specific planes

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.curvature import holonomy_rotor, holonomy_scalar_map

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

print("Imports OK")
print(f"NumPy {np.__version__}")

## Parallel Transport on the Grid

The layer-time grid has two axes. At each point $(l, t)$, we have a hidden-state vector $h^{(l)}_t \in \mathbb{R}^k$. Moving between adjacent grid points defines a **transport rotor** — the rotation that maps one hidden state's direction to the next.

A **plaquette** is the smallest closed loop on the grid: a unit square with corners at $(l, t)$, $(l+1, t)$, $(l+1, t+1)$, and $(l, t+1)$.

There are two paths from $(l, t)$ to $(l+1, t+1)$:

```
    (l, t+1) ──────── (l+1, t+1)
       |          ↗         |
       |   Path 2    Path 1 |
       |       ↗             |
    (l, t) ────────── (l+1, t)
```

**Path 1** (layer first): $(l,t) \to (l+1,t) \to (l+1,t+1)$
**Path 2** (token first): $(l,t) \to (l,t+1) \to (l+1,t+1)$

If space is flat, both paths produce the same result. If space is curved, they disagree. The **holonomy rotor** $R_{\text{loop}} = P_1 \cdot P_2^{-1}$ measures this disagreement.

Let us visualize this loop concept before computing it on real data.

In [ ]:
# ── Conceptual diagram: the plaquette loop ──

print("Plaquette at (l, t) — the holonomy loop")
print("=" * 50)
print()
print("  Corner points:")
print("    (l,   t  ) = bottom-left   [start]")
print("    (l+1, t  ) = bottom-right")
print("    (l+1, t+1) = top-right     [destination]")
print("    (l,   t+1) = top-left")
print()
print("  Path 1 (layer first, then token):")
print("    (l,t) --R_up--> (l+1,t) --R_right--> (l+1,t+1)")
print("    Composed rotor: P1 = R_right(l+1,t) * R_up(l,t)")
print()
print("  Path 2 (token first, then layer):")
print("    (l,t) --R_right--> (l,t+1) --R_up--> (l+1,t+1)")
print("    Composed rotor: P2 = R_up(l,t+1) * R_right(l,t)")
print()
print("  Holonomy rotor:")
print("    R_loop = P1 * P2^{-1}")
print()
print("  If R_loop = I (identity):  FLAT at this plaquette")
print("  If R_loop != I:            CURVED at this plaquette")
print()
print("  Scalar curvature:  ||R_loop - I||_F")
print("  Curvature bivector: log(R_loop)  -->  direction of curvature")

## The Holonomy Rotor

The holonomy rotor $R_{\text{loop}}$ at each plaquette encodes the local curvature. Let us unpack what it tells us:

**$R_{\text{loop}} = I$ (identity):** The two paths agree perfectly. The representation space is **flat** at this plaquette. Moving a vector around the loop brings it back to where it started.

**$R_{\text{loop}} \neq I$:** The two paths disagree. The space is **curved**. The rotor $R_{\text{loop}}$ tells us:

1. **Scalar curvature** = $\|R_{\text{loop}} - I\|_F$: How much the paths disagree (a single number, backward-compatible with non-GA approaches).

2. **Curvature bivector** = $\log(R_{\text{loop}})$: The bivector generator of the holonomy. Its principal plane tells you *in which 2D subspace* the curvature lives. This is the directional information that GA provides.

**Physical interpretation**: High curvature at plaquette $(l, t)$ means that the transformer's processing at layer $l$ depends strongly on the *token context* — what the model does to the representation at token $t$ versus token $t+1$ is qualitatively different, and this difference interacts with the layer-to-layer evolution. Flat regions indicate uniform, context-independent processing.

Let us compute and visualize this on a real model.

In [ ]:
# ── Load model and run GA analysis ──

model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Model: {model.name}")
print(f"  Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")
print()

# Analyse a prompt
prompt = "The cat sat on the mat and looked out the window at the birds"
rf = ltg_ga.analyse(prompt, model=model, compute_dependency=False)
print(f"Prompt: \"{prompt}\"")
print(f"  Tokens: {rf.n_tokens}, Whitened dim: {rf.k}")
print(f"  Holonomy map shape: {rf.holonomy_map.shape}")
print()

# ── Compute full holonomy scalar map ──
holo_map = rf.holonomy_map

print(f"Holonomy scalar curvature statistics:")
print(f"  Min:  {holo_map.min():.4f}")
print(f"  Max:  {holo_map.max():.4f}")
print(f"  Mean: {holo_map.mean():.4f}")
print(f"  Std:  {holo_map.std():.4f}")
print()

# ── Plot as heatmap ──
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(holo_map, aspect='auto', cmap='YlOrRd', origin='lower')
ax.set_xlabel('Token position', fontsize=12)
ax.set_ylabel('Layer', fontsize=12)
ax.set_title('Holonomy Scalar Curvature $\\|R_{\\mathrm{loop}} - I\\|_F$', fontsize=13)

# Add token labels if not too many
if rf.n_tokens <= 20:
    ax.set_xticks(range(rf.n_tokens - 1))
    ax.set_xticklabels([rf.tokens[t][:6] for t in range(rf.n_tokens - 1)],
                       rotation=45, ha='right', fontsize=8)

cbar = plt.colorbar(im, ax=ax, label='Scalar curvature', shrink=0.85)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch10_holonomy_map.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()
plt.close(fig)

## Curvature by Layer

The holonomy heatmap shows curvature at every plaquette. To see the big picture, we can average across the token axis to get the **mean curvature per layer**.

This profile reveals where the transformer's geometry is most curved — i.e., where the interaction between layer processing and token context is strongest. Typically:

- **Early layers** may show moderate curvature (initial feature extraction is context-dependent).
- **Middle layers** often show lower curvature (more uniform processing).
- **Late layers** may spike in curvature (context-dependent decision-making near the output).

The peak curvature layer is particularly interesting — it is where the transformer's processing is most "non-trivially geometric," meaning the order of operations (layer vs. token) matters the most.

In [ ]:
# ── Mean curvature per layer ──

curv_by_layer = rf.curvature_by_layer
n_layers = len(curv_by_layer)
peak_layer = int(np.argmax(curv_by_layer))

print(f"Curvature by layer ({n_layers} layers):")
print(f"  Peak curvature layer: {peak_layer}")
print(f"  Peak curvature value: {curv_by_layer[peak_layer]:.4f}")
print(f"  Mean curvature:       {curv_by_layer.mean():.4f}")
print()

# ── Plot: line + fill, mark peak layer ──
fig, ax = plt.subplots(figsize=(10, 5))
layers = np.arange(n_layers)

ax.plot(layers, curv_by_layer, color='#EE6677', linewidth=2.5, zorder=3)
ax.fill_between(layers, curv_by_layer, alpha=0.2, color='#EE6677')

# Mark the peak
ax.axvline(peak_layer, color='grey', linestyle='--', alpha=0.7,
           label=f'Peak layer = {peak_layer}')
ax.scatter([peak_layer], [curv_by_layer[peak_layer]], color='#CC3366',
           s=80, zorder=4, edgecolors='white', linewidth=1.5)

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Mean holonomy curvature', fontsize=12)
ax.set_title('Curvature Profile by Layer (GA Holonomy)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()

save_path = os.path.join(os.getcwd(), 'ch10_curvature_by_layer.png')
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()
plt.close(fig)

## The Curvature Direction

So far we have used $\|R_{\text{loop}} - I\|_F$ — a scalar. But the holonomy rotor carries much more information. Its bivector generator $\log(R_{\text{loop}})$ is a skew-symmetric matrix that can be decomposed into principal planes, just like any bivector.

The **principal plane of the curvature bivector** tells us which 2D subspace of the representation is most affected by the path-dependence. This is the *direction* of curvature.

**Why does this matter?** If two plaquettes have the same scalar curvature but different curvature directions, they represent fundamentally different geometric phenomena. One might be curving in a "syntactic" subspace while the other curves in a "semantic" subspace. The scalar measure cannot distinguish them; the bivector can.

Let us inspect the holonomy rotor at the plaquette with the highest scalar curvature.

In [ ]:
# ── Find the peak-curvature plaquette ──

# Get the (layer, token) index of the maximum curvature
peak_idx = np.unravel_index(np.argmax(holo_map), holo_map.shape)
peak_l, peak_t = int(peak_idx[0]), int(peak_idx[1])

print(f"Peak curvature plaquette: (layer={peak_l}, token={peak_t})")
print(f"  Scalar curvature: {holo_map[peak_l, peak_t]:.4f}")
print()

# ── Compute the full holonomy rotor at this plaquette ──
hr = holonomy_rotor(rf.H_whitened, peak_l, peak_t)

print(f"Holonomy rotor at peak plaquette:")
print(f"  Scalar curvature (||R_loop - I||_F): {hr.scalar_curvature:.4f}")
print(f"  Bivector norm (||log(R_loop)||_F):    {hr.bivector_norm:.4f}")
print(f"  Is identity: {hr.rotor.is_identity}")
print()

# ── Extract the principal plane of the curvature bivector ──
if hr.bivector is not None:
    planes = hr.bivector.principal_planes(n_planes=3)
    print("Principal planes of the curvature bivector:")
    print("-" * 50)
    for i, plane in enumerate(planes):
        print(f"  Plane {i+1}:")
        print(f"    Weight: {plane['weight']:.4f}")
        print(f"    Angle:  {plane['angle']:.4f} rad")
        v1, v2 = plane['plane_vectors']
        print(f"    v1:     [{', '.join(f'{x:.3f}' for x in v1[:5])}  ...]")
        print(f"    v2:     [{', '.join(f'{x:.3f}' for x in v2[:5])}  ...]")
        print()
    
    if planes:
        total_w = sum(p['weight'] for p in planes)
        top_frac = planes[0]['weight'] / total_w * 100 if total_w > 0 else 0
        print(f"  Dominant plane concentration: {top_frac:.1f}% of total weight")
        if top_frac > 60:
            print("  --> Curvature is concentrated in a single plane")
        else:
            print("  --> Curvature is spread across multiple planes")
else:
    print("  (no bivector — plaquette is flat)")

## Do Different Plaquettes Curve in the Same Direction?

This is perhaps the most profound question we can ask with the GA framework: **is the curvature across the layer-time grid geometrically coherent?**

If the curvature bivectors at different high-curvature plaquettes all point in approximately the same plane, this means the transformer has a single dominant "curvature mode" — the path-dependence always affects the same subspace of the representation. This would suggest a structured, low-dimensional geometric phenomenon.

If the curvature directions are all different, the geometry is more complex — different parts of the grid curve in different subspaces, suggesting multiple independent geometric processes.

To compare curvature directions, we look at the principal planes of the curvature bivectors at the top-5 highest-curvature plaquettes and compare their angles and directions. This analysis is *only possible* with the GA framework — a scalar curvature map gives no information about directional consistency.

In [ ]:
# ── Compare curvature directions at the top-5 highest-curvature plaquettes ──

# Flatten the heatmap and get the top-5 indices
n_top = 5
flat_indices = np.argsort(holo_map.ravel())[::-1][:n_top]
top_plaquettes = [np.unravel_index(idx, holo_map.shape) for idx in flat_indices]

print(f"Top-{n_top} Highest-Curvature Plaquettes")
print("=" * 65)

# Store dominant plane vectors for comparison
dominant_v1s = []

for rank, (l, t) in enumerate(top_plaquettes):
    l, t = int(l), int(t)
    hr = holonomy_rotor(rf.H_whitened, l, t)
    
    print(f"\n  #{rank+1}: Plaquette (layer={l}, token={t})")
    print(f"    Scalar curvature: {hr.scalar_curvature:.4f}")
    print(f"    Bivector norm:    {hr.bivector_norm:.4f}")
    
    if hr.bivector is not None:
        planes = hr.bivector.principal_planes(n_planes=1)
        if planes:
            p = planes[0]
            print(f"    Dominant plane angle: {p['angle']:.4f} rad")
            print(f"    Dominant plane weight: {p['weight']:.4f}")
            dominant_v1s.append(p['plane_vectors'][0])
        else:
            print(f"    (no significant planes)")
            dominant_v1s.append(None)
    else:
        print(f"    (flat — no bivector)")
        dominant_v1s.append(None)

# ── Compare the dominant plane directions across plaquettes ──
print("\n\nDirectional Consistency Analysis")
print("=" * 65)
print("Cosine similarity between dominant plane vectors (v1) of top plaquettes:\n")

valid_vs = [(i, v) for i, v in enumerate(dominant_v1s) if v is not None]
if len(valid_vs) >= 2:
    print(f"{'':>10}", end="")
    for i, _ in valid_vs:
        print(f"  #{i+1:>5}", end="")
    print()
    
    for i, vi in valid_vs:
        print(f"  #{i+1:>5}  ", end="")
        for j, vj in valid_vs:
            cos_sim = abs(np.dot(vi, vj))  # abs because ±v define the same plane
            print(f"  {cos_sim:5.3f}", end="")
        print()
    
    # Average pairwise similarity
    sims = []
    for idx_a in range(len(valid_vs)):
        for idx_b in range(idx_a + 1, len(valid_vs)):
            cos = abs(np.dot(valid_vs[idx_a][1], valid_vs[idx_b][1]))
            sims.append(cos)
    
    mean_sim = np.mean(sims)
    print(f"\n  Mean pairwise |cos| similarity: {mean_sim:.3f}")
    if mean_sim > 0.7:
        print("  --> HIGH coherence: curvature points in a consistent direction")
    elif mean_sim > 0.3:
        print("  --> MODERATE coherence: some directional consistency")
    else:
        print("  --> LOW coherence: curvature directions are largely independent")
else:
    print("  Not enough plaquettes with significant curvature for comparison.")

## The Nonseparability Index and Curvature Regimes

The holonomy map gives curvature at every plaquette. To summarise the **total** amount
of non-trivial computation in a single number, we sum it:

$$\mathcal{D}(s) = \sum_{l,t} \|\Omega^{(l,t)}\|_F$$

- $\mathcal{D} = 0$: **separable** — layer and token operations are independent (simple lookup)
- $\mathcal{D} > 0$: **interactive** — which token you are changes what the layer does

We can classify prompts into **curvature regimes**:

| Regime | Signature | Meaning |
|--------|-----------|---------|
| Flat | $\|\Omega\| \approx 0$ everywhere | Memorisation / lookup |
| Low | $\|\Omega\|$ small, smooth | Straightforward recall |
| High | $\|\Omega\|$ large, structured | Multi-step reasoning |
| Chaotic | $\mathrm{Var}(\|\Omega\|) \gg 0$ | Potential hallucination |

In [ ]:
# ── Compute nonseparability index and classify curvature regime ───
from layer_time_ga.curvature import nonseparability_index

ns = nonseparability_index(rf.H_whitened)

print("Nonseparability Index")
print("=" * 40)
print(f"  D(s) total:    {ns['D_total']:.2f}")
print(f"  D(s) mean:     {ns['D_mean']:.4f}")
print(f"  Regime:        {ns['regime']}")
print(f"  Grid shape:    {ns['holo_map'].shape}")

# Fraction of curvature in final 3 layers
hmap = ns['holo_map']
n_layers = hmap.shape[0]
final_3 = hmap[-3:].sum()
print(f"\n  Curvature in final 3 layers: {final_3/ns['D_total']*100:.1f}%")

# Coefficient of variation
cv = float(np.std(hmap) / (ns['D_mean'] + 1e-12))
print(f"  Coefficient of variation:    {cv:.2f}")
print(f"\n→ Regime '{ns['regime']}' means the model is performing "
      f"{'interactive' if ns['regime'] in ('high','low') else 'trivial or unstable'} "
      f"computation.")

## Exercises

**Exercise 10.1 — Compare two prompts.**
Run the holonomy analysis on two different prompts — one factual (e.g., "The capital of France is Paris") and one more abstract (e.g., "Beauty is truth, truth beauty"). Compare their:
- Scalar curvature heatmaps
- Peak curvature layers
- Mean curvature profiles

Does the abstract prompt produce more curvature? Does the peak shift?

**Exercise 10.2 — Curvature plane consistency across tokens.**
Fix a single layer (the peak curvature layer) and compute the holonomy rotor at every token position along that layer. Extract the dominant curvature plane at each token. Are the curvature planes consistent across tokens within the same layer, or do different tokens curve in different directions? Plot the dominant plane angle as a function of token position.

**Exercise 10.3 — Total curvature.**
Define the **total curvature** as the sum of all scalar curvature values across the heatmap: $\kappa_{\text{total}} = \sum_{l,t} \|R_{\text{loop}}(l,t) - I\|_F$. Compute this for your prompt. Then normalize by the number of plaquettes to get the **mean curvature density**. How does this relate to the model's ability to contextualize representations?

**Exercise 10.4 — Flat regions.**
Identify the flattest region of the heatmap (the rectangular block with the lowest mean curvature). What layers and tokens does it correspond to? What might it mean for a region to be geometrically flat — does the transformer do "nothing interesting" there, or could it be doing something important that happens to be path-independent?

**Exercise 10.5 — Holonomy and the commutator (connecting to Chapter 9).**
The holonomy rotor is a *finite* measure of curvature (composing rotors around a loop), while the commutator $[B_i, B_j]$ is an *infinitesimal* measure (how much two generators fail to commute). At the same layer, compare the holonomy scalar curvature (averaged over tokens) with the commutator norm $\|[B_l, B_{l+1}]\|_F$ for adjacent layers. Are they correlated? Plot both on the same graph and discuss.